Libraries:

In [ ]:
import pandas as pd
import numpy as np
import os

Firts of all, we should have all the .FITS files correctly downloaded. 

Our first step is getting the matrix with all the SNR values for each filter. All the code needed for this in in the file SNR_values.py in the same folder.

In [ ]:
from SNR_PhoCount import matriz, fits_paths

data = '../data/'
paths = fits_paths(data)
matrix = matriz(paths)
print(matrix)

For the excel file crafting we first set the paths

In [ ]:
base_path = os.getcwd() 
catalog_path = os.path.join(base_path, 'catalog.xlsx')
output_path = os.path.join(base_path, '..', 'data', 'processed', 'SNR_and_count_values.xlsx')

print(f"Current Working Directory: {base_path}")

Now we create a sheet in an excel file with the information of the SNR values and the respective stellar source information form the original catalog. 

In [ ]:
snr_list = []
count_list = []

for image_id, snr_vals, count_vals in matrix:
    # Prepare rows for both DataFrames
    snr_row = {'Image': image_id}
    count_row = {'Image': image_id}
    
    for i in range(3):
        col_snr = f'SNR Filter {i+1}'
        col_count = f'Count Filter {i+1}'
        
        snr_row[col_snr] = np.nan if snr_vals[i] == 'negative' else snr_vals[i]
        
        count_row[col_count] = np.nan if count_vals[i] == 'negative' else count_vals[i]
        
    snr_list.append(snr_row)
    count_list.append(count_row)

df_snr_raw = pd.DataFrame(snr_list)
df_count_raw = pd.DataFrame(count_list)

# Fixing the Image column
df_catalog = pd.read_excel(catalog_path)
df_catalog['Image'] = 'SWP' + df_catalog['Image'].astype(str).str.zfill(5)

# 'Cl' coulumn (Most frequent, then max if tie)
def cl_logic(series):
    # .mode() returns a Series of the most frequent values
    modes = series.mode()
    if not modes.empty:
        # In case of a tie, .max() picks the highest value
        return modes.max() 
    return None

cl_mapping = df_catalog.groupby('Object')['Cl'].apply(cl_logic).reset_index()
# Replace original Cl with the calculated representative Cl
df_catalog = df_catalog.drop(columns=['Cl']).merge(cl_mapping, on='Object')

# 5. Merge and Prepare Final DataFrames
common_cols = ['Object', 'Cl', 'RA(1950)', 'Dec(1950)', 'Image', 'ExpTime']

df_final_snr = pd.merge(df_catalog[common_cols], df_snr_raw, on='Image', how='inner').sort_values(['Object', 'Image'])
df_final_count = pd.merge(df_catalog[common_cols], df_count_raw, on='Image', how='inner').sort_values(['Object', 'Image'])

# 6. Export to Excel (Multiple Sheets)
print(f"Generating final excel at: {output_path}")

with pd.ExcelWriter(output_path, engine='xlsxwriter') as writer:
    # Write Sheets
    df_final_snr.to_excel(writer, index=False, sheet_name='SNR_Results')
    df_final_count.to_excel(writer, index=False, sheet_name='Count_Results')
    

print("Process completed successfully.")

**analysis**
Now we can work on the analysis of this data, we can either do this directly on excel or get the values with code.

We are getting the mean photons counting per filter and object and its standard deviation.

In [ ]:
count_columns = ['Count Filter 1', 'Count Filter 2', 'Count Filter 3']

# We take the first value for Cl, RA(1950), Dec(1950), since they are the same for each object
agg_logic = {
    'Cl': 'first',
    'RA(1950)': 'first',
    'Dec(1950)': 'first',
    'Image': 'count'
}

for col in count_columns:
    agg_logic[col] = ['mean', 'std']

df_stats = df_final_count.groupby('Object').agg(agg_logic)
df_stats.columns = [
    'Cl', 'RA(1950)', 'Dec(1950)', 'Image_Count',
    'Mean_F1', 'Std_F1', 'Mean_F2', 'Std_F2', 'Mean_F3', 'Std_F3'
]
df_stats = df_stats.reset_index()
ordered_cols = [
    'Object', 'Cl', 'RA(1950)', 'Dec(1950)', 'Image_Count',
    'Mean_F1', 'Mean_F2', 'Mean_F3', 
    'Std_F1', 'Std_F2', 'Std_F3'
]
df_stats = df_stats[ordered_cols]
print(f"Updating excel with stats sheet at: {output_path}")

with pd.ExcelWriter(output_path, engine='xlsxwriter') as writer:
    
    df_final_snr.to_excel(writer, index=False, sheet_name='SNR_Results')
    df_final_count.to_excel(writer, index=False, sheet_name='Count_Results')
    df_stats.to_excel(writer, index=False, sheet_name='Stats_Summary')

    for sheet_name in writer.sheets:
        worksheet = writer.sheets[sheet_name]
        if sheet_name == 'SNR_Results': temp_df = df_final_snr
        elif sheet_name == 'Count_Results': temp_df = df_final_count
        else: temp_df = df_stats
        
        for i, col in enumerate(temp_df.columns):
            max_len = max(temp_df[col].astype(str).str.len().max(), len(col)) + 2
            worksheet.set_column(i, i, max_len)